# Catastrophic Forgetting Evaluation
**4M-21 XL — Baseline vs LoRA (depth fine-tuning, rank=8)**

Tests whether LoRA fine-tuning on depth estimation degrades performance on other modalities:
- **Section A** — RGB Reconstruction (NYU + DIODE)
- **Section B** — Surface Normals (DIODE)
- **Section C** — Semantic Segmentation (NYU)

Output saved to `output/eval_forgetting/`.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
import os
REPO_ROOT    = '/home/fcolasurdo/4m-peft'
os.chdir(REPO_ROOT)

FM_MODEL     = 'EPFL-VILAB/4M-21_XL'
LORA_WEIGHTS = 'output/depth/nyu_rgb_r8/lora_weights-epoch49.pth'
LORA_RANK    = 8
LORA_MODULES = 'attention'

HF_CACHE     = '/scratch-shared/tesr125338/hf_cache'
TOK_DEPTH    = 'EPFL-VILAB/4M_tokenizers_depth_8k_224-448'
TOK_NORMAL   = 'EPFL-VILAB/4M_tokenizers_normal_8k_224-448'
TOK_SEMSEG   = 'EPFL-VILAB/4M_tokenizers_semseg_4k_224-448'
TOK_RGB      = 'EPFL-VILAB/4M_tokenizers_rgb_16k_224-448'
TOK_TEXT     = 'ml-4m/fourm/utils/tokenizer/trained/text_tokenizer_4m_wordpiece_30k.json'

DIODE_ROOT   = '/scratch-shared/tesr125338/diode/val'
NYU_MAT      = '/scratch-shared/tesr125338/nyu_depth_v2_labeled.mat'
NYU_RAW      = '/scratch-shared/tesr125338/nyu_raw/test'

OUTPUT_DIR   = 'output/eval_forgetting'
N_SAMPLES    = 8    # images per section (visual grid)
N_EVAL       = 50   # images for metric computation
SEED         = 42
DIVAE_STEPS  = 25

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys, random, json
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from PIL import Image
from tqdm import tqdm

sys.path.insert(0, os.path.join(REPO_ROOT, 'ml-4m'))

from fourm.models.fm import FM
from fourm.vq.vqvae import DiVAE, VQVAE
from fourm.models.generate import GenerationSampler, build_chained_generation_schedules
from fourm.utils.generation import init_full_input_modality, init_empty_target_modality
from fourm.utils import MODALITY_INFO
from fourm.models.lora_utils import inject_trainable_LoRA, get_LoRA_module_names
from tokenizers import Tokenizer

import torchvision.transforms as T

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ── Load models ───────────────────────────────────────────────────────────────
print('Loading baseline FM ...')
fm_base = FM.from_pretrained(FM_MODEL, cache_dir=HF_CACHE).to(device).eval()

print('Loading LoRA FM ...')
fm_lora = FM.from_pretrained(FM_MODEL, cache_dir=HF_CACHE)
target_modules = get_LoRA_module_names(LORA_MODULES)
inject_trainable_LoRA(fm_lora, rank=LORA_RANK, scale=1.0, target_replace_modules=target_modules)
lora_sd = torch.load(LORA_WEIGHTS, map_location='cpu')
missing, unexpected = fm_lora.load_state_dict(lora_sd, strict=False)
print(f'  Loaded {len(lora_sd)} LoRA tensors | missing={len(missing)}')
fm_lora = fm_lora.to(device).eval()

print('Loading tokenizers ...')
tok_depth  = DiVAE.from_pretrained(TOK_DEPTH,  cache_dir=HF_CACHE).to(device).eval()
tok_normal = DiVAE.from_pretrained(TOK_NORMAL, cache_dir=HF_CACHE).to(device).eval()
tok_semseg = VQVAE.from_pretrained(TOK_SEMSEG, cache_dir=HF_CACHE).to(device).eval()
tok_rgb    = VQVAE.from_pretrained(TOK_RGB,    cache_dir=HF_CACHE).to(device).eval()
tok_text   = Tokenizer.from_file(os.path.join(REPO_ROOT, TOK_TEXT))

sampler_base = GenerationSampler(fm_base)
sampler_lora = GenerationSampler(fm_lora)
print('All models ready.')

In [ ]:
# ── Shared generation params ──────────────────────────────────────────────────
# Copy from viz_depth_results.ipynb — same params, only target_domain changes
GEN_PARAMS = dict(
    tokens_per_target        = 196,
    autoregression_scheme    = 'roar',
    decoding_steps           = 25,
    token_decoding_schedule  = 'linear',
    temp                     = 1.0,
    temp_schedule            = 'constant',
    cfg_scale                = 2.0,
    cfg_schedule             = 'constant',
)

def run_inference_modality(sampler, rgb_tensor, target_mod, seed=42):
    """Generic inference: RGB → any modality. Returns raw decoded output."""
    ntoks    = GEN_PARAMS['tokens_per_target']
    tok_mod  = f'tok_{target_mod}@224'   # e.g. 'tok_normal@224'
    cond_mod = 'rgb@224'
    schedule = build_chained_generation_schedules(
        cond_domains=[cond_mod], target_domains=[tok_mod],
        tokens_per_target=[ntoks],
        autoregression_schemes=[GEN_PARAMS['autoregression_scheme']],
        decoding_steps=[GEN_PARAMS['decoding_steps']],
        token_decoding_schedules=[GEN_PARAMS['token_decoding_schedule']],
        temps=[GEN_PARAMS['temp']], temp_schedules=[GEN_PARAMS['temp_schedule']],
        cfg_scales=[GEN_PARAMS['cfg_scale']], cfg_schedules=[GEN_PARAMS['cfg_schedule']],
        cfg_grow_conditioning=True,
    )
    eos_id = tok_text.token_to_id('[EOS]')
    sd = {cond_mod: {'tensor': rgb_tensor}}
    sd = init_full_input_modality(sd, MODALITY_INFO, cond_mod, device, eos_id=eos_id)
    sd = init_empty_target_modality(sd, MODALITY_INFO, tok_mod, 1, ntoks, device)
    out = sampler.generate(sd, schedule, text_tokenizer=tok_text, verbose=False, seed=seed)
    return out, tok_mod

# ── Shared helpers ─────────────────────────────────────────────────────────────
rgb_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def load_rgb(path):
    img = Image.open(path).convert('RGB')
    tensor = rgb_transform(img).unsqueeze(0).to(device)
    rgb_np = np.array(img.resize((224, 224))) / 255.0
    return rgb_np, tensor

def save_fig(fig, name):
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Saved → {path}')

print('Helper functions ready.')

---
## Section A — RGB Reconstruction
**Task**: tok_rgb@224 → tok_rgb@224  
**Question**: Does LoRA degrade the model's basic visual reconstruction ability?  
**Metrics**: SSIM, PSNR  
**Datasets**: NYU + DIODE

In [ ]:
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# ── Load samples ──────────────────────────────────────────────────────────────
nyu_rgb_files  = sorted(Path(NYU_RAW).glob('rgb/*.png'))[:N_SAMPLES]
diode_rgb_files = sorted(Path(DIODE_ROOT).rglob('*.png'))[:N_SAMPLES]
print(f'NYU samples:   {len(nyu_rgb_files)}')
print(f'DIODE samples: {len(diode_rgb_files)}')

def decode_rgb_tokens(out, tok_mod):
    """Decode predicted RGB tokens → numpy (H,W,3) in [0,1]."""
    tokens = out[tok_mod]['tokens'].squeeze(0)   # (196,)
    tokens_2d = tokens.reshape(1, 14, 14)
    with torch.no_grad():
        decoded = tok_rgb.decode_tokens(tokens_2d, image_size=224)
    img = decoded.squeeze(0).permute(1,2,0).cpu().numpy()
    return np.clip((img + 1) / 2, 0, 1)  # [-1,1] → [0,1]

def eval_rgb_reconstruction(sampler, rgb_files, label, n=N_EVAL):
    ssims, psnrs = [], []
    for p in tqdm(rgb_files[:n], desc=label):
        try:
            rgb_np, rgb_t = load_rgb(p)
            with torch.no_grad():
                out, tok_mod = run_inference_modality(sampler, rgb_t, 'rgb', seed=SEED)
            pred = decode_rgb_tokens(out, tok_mod)
            ssims.append(ssim(rgb_np, pred, channel_axis=2, data_range=1.0))
            psnrs.append(psnr(rgb_np, pred, data_range=1.0))
        except Exception as e:
            print(f'  skip {p.name}: {e}')
    print(f'  SSIM={np.mean(ssims):.4f}  PSNR={np.mean(psnrs):.2f} dB  N={len(ssims)}')
    return ssims, psnrs

print('=== NYU — RGB Reconstruction ===')
print('Baseline:'); nyu_base_ssim, nyu_base_psnr = eval_rgb_reconstruction(sampler_base, nyu_rgb_files, 'NYU baseline')
print('LoRA:');     nyu_lora_ssim, nyu_lora_psnr = eval_rgb_reconstruction(sampler_lora, nyu_rgb_files, 'NYU lora')

print('=== DIODE — RGB Reconstruction ===')
print('Baseline:'); dio_base_ssim, dio_base_psnr = eval_rgb_reconstruction(sampler_base, diode_rgb_files, 'DIODE baseline')
print('LoRA:');     dio_lora_ssim, dio_lora_psnr = eval_rgb_reconstruction(sampler_lora, diode_rgb_files, 'DIODE lora')

In [ ]:
# ── Visualize RGB reconstruction ──────────────────────────────────────────────
def viz_rgb_reconstruction(rgb_files, sampler_b, sampler_l, title, save_name, n=N_SAMPLES):
    files = rgb_files[:n]
    fig, axes = plt.subplots(len(files), 3, figsize=(10, len(files)*3))
    for col, lbl in enumerate(['Input RGB', 'Baseline recon', 'LoRA recon']):
        axes[0, col].set_title(lbl, fontsize=11, fontweight='bold')
    for row, p in enumerate(files):
        rgb_np, rgb_t = load_rgb(p)
        with torch.no_grad():
            out_b, tm = run_inference_modality(sampler_b, rgb_t, 'rgb', seed=SEED+row)
            out_l, _  = run_inference_modality(sampler_l, rgb_t, 'rgb', seed=SEED+row)
        pred_b = decode_rgb_tokens(out_b, tm)
        pred_l = decode_rgb_tokens(out_l, tm)
        s_b = ssim(rgb_np, pred_b, channel_axis=2, data_range=1.0)
        s_l = ssim(rgb_np, pred_l, channel_axis=2, data_range=1.0)
        axes[row,0].imshow(rgb_np);  axes[row,0].axis('off')
        axes[row,1].imshow(pred_b); axes[row,1].axis('off')
        axes[row,1].set_xlabel(f'SSIM={s_b:.3f}', fontsize=8, color='#1a6bb5', fontweight='bold')
        better = s_l > s_b
        axes[row,2].imshow(pred_l); axes[row,2].axis('off')
        axes[row,2].set_xlabel(
            f'SSIM={s_l:.3f} ({"▲" if better else "▼"})',
            fontsize=8, color='#2a9d2a' if better else '#c0392b', fontweight='bold')
    fig.suptitle(title, fontsize=12, y=1.01)
    plt.tight_layout()
    save_fig(fig, save_name)
    plt.show()

viz_rgb_reconstruction(nyu_rgb_files,  sampler_base, sampler_lora,
                       'RGB Reconstruction — NYU', 'rgb_recon_nyu.png')
viz_rgb_reconstruction(diode_rgb_files, sampler_base, sampler_lora,
                       'RGB Reconstruction — DIODE', 'rgb_recon_diode.png')

---
## Section B — Surface Normals
**Task**: tok_rgb@224 → tok_normal@224  
**Question**: Does LoRA on depth degrade surface normal prediction (geometrically related task)?  
**Metrics**: Mean Angular Error (MAE °), δ<11.25°, δ<22.5°, δ<30°  
**Dataset**: DIODE val (indoor + outdoor, normals in `*_normal.npy`)

In [ ]:
# ── Load DIODE samples with normals ───────────────────────────────────────────
diode_samples_normal = []
for rgb_p in sorted(Path(DIODE_ROOT).rglob('*.png')):
    normal_p = rgb_p.parent / (rgb_p.stem + '_normal.npy')
    mask_p   = rgb_p.parent / (rgb_p.stem + '_depth_mask.npy')
    if normal_p.exists() and mask_p.exists():
        diode_samples_normal.append({'rgb': rgb_p, 'normal': normal_p, 'mask': mask_p})

print(f'DIODE samples with normals: {len(diode_samples_normal)}')
if len(diode_samples_normal) == 0:
    print('WARNING: no normal files found — download val_normals.tar.gz first')
    print('  cd /scratch-shared/tesr125338/diode')
    print('  wget http://diode-dataset.s3.amazonaws.com/val_normals.tar.gz')
    print('  tar -xzf val_normals.tar.gz && rm val_normals.tar.gz')

In [ ]:
def decode_normal_tokens(out, tok_mod):
    """Decode predicted normal tokens → numpy (H,W,3) unit vectors."""
    tokens = out[tok_mod]['tokens'].squeeze(0).reshape(1, 14, 14)
    with torch.no_grad():
        decoded = tok_normal.decode_tokens(tokens, timesteps=DIVAE_STEPS,
                                           image_size=224, verbose=False)
    n = decoded.squeeze(0).permute(1,2,0).cpu().numpy()  # (224,224,3)
    # normalise to unit vectors
    norm = np.linalg.norm(n, axis=-1, keepdims=True)
    return n / (norm + 1e-8)

def angular_error(pred_n, gt_n, valid):
    """Mean angular error in degrees between predicted and GT normals."""
    dot = np.clip((pred_n[valid] * gt_n[valid]).sum(-1), -1, 1)
    angles = np.degrees(np.arccos(dot))
    return angles

def eval_normals(sampler, samples, label, n=N_EVAL):
    all_angles = []
    for s in tqdm(samples[:n], desc=label):
        try:
            rgb_np, rgb_t = load_rgb(s['rgb'])
            gt_normal = np.load(s['normal'])   # (H,W,3)
            mask = np.load(s['mask']).astype(bool).squeeze()
            # resize GT to 224
            gt_r = np.array(Image.fromarray(
                ((gt_normal+1)/2*255).astype(np.uint8)).resize((224,224))) / 255.0 * 2 - 1
            norm = np.linalg.norm(gt_r, axis=-1, keepdims=True)
            gt_r = gt_r / (norm + 1e-8)
            valid = mask[:224, :224] if mask.shape[0] >= 224 else np.ones((224,224), bool)
            with torch.no_grad():
                out, tok_mod = run_inference_modality(sampler, rgb_t, 'normal', seed=SEED)
            pred_n = decode_normal_tokens(out, tok_mod)
            angles = angular_error(pred_n, gt_r, valid)
            all_angles.extend(angles.tolist())
        except Exception as e:
            print(f'  skip {s["rgb"].name}: {e}')
    all_angles = np.array(all_angles)
    print(f'  MAE={all_angles.mean():.2f}°  '
          f'δ<11.25°={np.mean(all_angles<11.25):.4f}  '
          f'δ<22.5°={np.mean(all_angles<22.5):.4f}  '
          f'δ<30°={np.mean(all_angles<30):.4f}  N={len(all_angles)//196}')
    return all_angles

if len(diode_samples_normal) > 0:
    print('=== Surface Normals — DIODE ===')
    print('Baseline:'); norm_base = eval_normals(sampler_base, diode_samples_normal, 'baseline')
    print('LoRA:');     norm_lora = eval_normals(sampler_lora, diode_samples_normal, 'lora')
else:
    print('Skipping — download normals first (see cell above)')

In [ ]:
# ── Visualize surface normals ─────────────────────────────────────────────────
def normal_to_rgb(n):
    """Convert normal vectors (H,W,3) in [-1,1] to RGB image [0,1]."""
    return np.clip((n + 1) / 2, 0, 1)

if len(diode_samples_normal) > 0:
    samples_viz = diode_samples_normal[:N_SAMPLES]
    fig, axes = plt.subplots(len(samples_viz), 4, figsize=(13, len(samples_viz)*3))
    for col, lbl in enumerate(['RGB', 'GT Normal', 'Baseline', 'LoRA']):
        axes[0,col].set_title(lbl, fontsize=11, fontweight='bold')

    for row, s in enumerate(samples_viz):
        rgb_np, rgb_t = load_rgb(s['rgb'])
        gt_n = np.load(s['normal'])
        gt_r = np.array(Image.fromarray(((gt_n+1)/2*255).astype(np.uint8)).resize((224,224)))/255.0*2-1
        gt_r = gt_r / (np.linalg.norm(gt_r, axis=-1, keepdims=True) + 1e-8)

        with torch.no_grad():
            out_b, tm = run_inference_modality(sampler_base, rgb_t, 'normal', seed=SEED+row)
            out_l, _  = run_inference_modality(sampler_lora, rgb_t, 'normal', seed=SEED+row)
        pred_b = decode_normal_tokens(out_b, tm)
        pred_l = decode_normal_tokens(out_l, tm)

        valid = np.ones((224,224), bool)
        mae_b = angular_error(pred_b, gt_r, valid).mean()
        mae_l = angular_error(pred_l, gt_r, valid).mean()
        better = mae_l < mae_b

        axes[row,0].imshow(rgb_np);             axes[row,0].axis('off')
        axes[row,1].imshow(normal_to_rgb(gt_r));axes[row,1].axis('off')
        axes[row,2].imshow(normal_to_rgb(pred_b));axes[row,2].axis('off')
        axes[row,2].set_xlabel(f'MAE={mae_b:.1f}°', fontsize=8, color='#1a6bb5', fontweight='bold')
        axes[row,3].imshow(normal_to_rgb(pred_l));axes[row,3].axis('off')
        axes[row,3].set_xlabel(
            f'MAE={mae_l:.1f}° ({"▲ better" if better else "▼ worse"})',
            fontsize=8, color='#2a9d2a' if better else '#c0392b', fontweight='bold')

    fig.suptitle('Surface Normals — DIODE val\nColour = (normal+1)/2', fontsize=11, y=1.01)
    plt.tight_layout()
    save_fig(fig, 'normals_diode.png')
    plt.show()

---
## Section C — Semantic Segmentation
**Task**: tok_rgb@224 → tok_semseg@224  
**Question**: Does LoRA on depth degrade semantic understanding (unrelated task)?  
**Metrics**: mIoU (40 classes, NYU standard)  
**Dataset**: NYU Depth v2 (labels from nyu_depth_v2_labeled.mat)

In [ ]:
import scipy.io

# ── Load NYU semseg GT ────────────────────────────────────────────────────────
# Eigen test split indices (654 images)
EIGEN_TEST_INDICES = [
      1,    2,    3,    4,    5,    6,    7,    8,    9,   10,
     11,   12,   13,   14,   15,   16,   17,   18,   19,   20,
    # ... (full list below)
]
# Full Eigen test split (654 indices, 1-based)
import urllib.request
# Load from local file if available, else use first 654 images
try:
    mat = scipy.io.loadmat(NYU_MAT)
    labels_all = mat['labels']        # (H, W, N) uint16
    images_all = mat['images']        # (H, W, 3, N) uint8
    print(f'NYU mat loaded: {labels_all.shape[2]} images, {labels_all.max()} classes')
    # Use last 654 images as test split (Eigen convention)
    n_total = labels_all.shape[2]
    test_indices = list(range(n_total - 654, n_total))
    print(f'Test split: {len(test_indices)} images')
except Exception as e:
    print(f'Error loading mat: {e}')

In [ ]:
def decode_semseg_tokens(out, tok_mod):
    """Decode predicted semseg tokens → class map (H,W) int."""
    tokens = out[tok_mod]['tokens'].squeeze(0).reshape(1, 14, 14)
    with torch.no_grad():
        decoded = tok_semseg.decode_tokens(tokens, image_size=224)
    # decoded is (1, C, H, W) — take argmax over classes
    if decoded.dim() == 4:
        return decoded.squeeze(0).argmax(0).cpu().numpy()   # (H,W)
    return decoded.squeeze(0).cpu().numpy().astype(int)

def compute_miou(pred, gt, n_classes=40):
    ious = []
    for c in range(1, n_classes + 1):  # 1-indexed in NYU
        pred_c = pred == c
        gt_c   = gt == c
        inter  = (pred_c & gt_c).sum()
        union  = (pred_c | gt_c).sum()
        if union > 0:
            ious.append(inter / union)
    return np.mean(ious) if ious else 0.0

def eval_semseg(sampler, indices, label, n=N_EVAL):
    mious = []
    for idx in tqdm(indices[:n], desc=label):
        try:
            # Load RGB from mat
            img_np = images_all[:, :, :, idx].transpose(0,1,2)  # (H,W,3)
            img_pil = Image.fromarray(img_np)
            _, rgb_t = load_rgb_from_pil(img_pil)
            gt_label = labels_all[:, :, idx]  # (H,W)
            gt_r = np.array(Image.fromarray(gt_label.astype(np.uint8)).resize((224,224),
                            resample=Image.NEAREST))
            with torch.no_grad():
                out, tok_mod = run_inference_modality(sampler, rgb_t, 'semseg', seed=SEED)
            pred = decode_semseg_tokens(out, tok_mod)
            mious.append(compute_miou(pred, gt_r))
        except Exception as e:
            print(f'  skip idx {idx}: {e}')
    print(f'  mIoU={np.mean(mious):.4f}  N={len(mious)}')
    return mious

def load_rgb_from_pil(img_pil):
    tensor = rgb_transform(img_pil).unsqueeze(0).to(device)
    rgb_np = np.array(img_pil.resize((224,224))) / 255.0
    return rgb_np, tensor

print('=== Semantic Segmentation — NYU ===')
print('Baseline:'); seg_base = eval_semseg(sampler_base, test_indices, 'baseline')
print('LoRA:');     seg_lora = eval_semseg(sampler_lora, test_indices, 'lora')

In [ ]:
# ── Visualize semantic segmentation ──────────────────────────────────────────
cmap_seg = plt.get_cmap('tab20', 40)

viz_indices = test_indices[:N_SAMPLES]
fig, axes = plt.subplots(len(viz_indices), 4, figsize=(13, len(viz_indices)*3))
for col, lbl in enumerate(['RGB', 'GT Semseg', 'Baseline', 'LoRA']):
    axes[0,col].set_title(lbl, fontsize=11, fontweight='bold')

for row, idx in enumerate(viz_indices):
    img_np = images_all[:,:,:,idx].transpose(0,1,2)
    img_pil = Image.fromarray(img_np)
    rgb_np, rgb_t = load_rgb_from_pil(img_pil)
    gt_label = labels_all[:,:,idx]
    gt_r = np.array(Image.fromarray(gt_label.astype(np.uint8)).resize((224,224), resample=Image.NEAREST))

    with torch.no_grad():
        out_b, tm = run_inference_modality(sampler_base, rgb_t, 'semseg', seed=SEED+row)
        out_l, _  = run_inference_modality(sampler_lora, rgb_t, 'semseg', seed=SEED+row)
    pred_b = decode_semseg_tokens(out_b, tm)
    pred_l = decode_semseg_tokens(out_l, tm)

    miou_b = compute_miou(pred_b, gt_r)
    miou_l = compute_miou(pred_l, gt_r)
    better = miou_l > miou_b

    axes[row,0].imshow(rgb_np);                     axes[row,0].axis('off')
    axes[row,1].imshow(cmap_seg(gt_r/40.0));        axes[row,1].axis('off')
    axes[row,2].imshow(cmap_seg(pred_b/40.0));      axes[row,2].axis('off')
    axes[row,2].set_xlabel(f'mIoU={miou_b:.3f}', fontsize=8, color='#1a6bb5', fontweight='bold')
    axes[row,3].imshow(cmap_seg(pred_l/40.0));      axes[row,3].axis('off')
    axes[row,3].set_xlabel(
        f'mIoU={miou_l:.3f} ({"▲ better" if better else "▼ worse"})',
        fontsize=8, color='#2a9d2a' if better else '#c0392b', fontweight='bold')

fig.suptitle('Semantic Segmentation — NYU (40 classes)', fontsize=11, y=1.01)
plt.tight_layout()
save_fig(fig, 'semseg_nyu.png')
plt.show()

---
## Summary

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print('=' * 65)
print('CATASTROPHIC FORGETTING EVALUATION SUMMARY')
print('LoRA: depth fine-tuning rank=8, epoch 49')
print('=' * 65)
print('{:<30} {:>10} {:>10} {:>8}'.format('Task / Metric', 'Baseline', 'LoRA', 'Δ'))
print('-' * 65)

def delta_str(b, l, higher_better=True):
    d = l - b
    sign = '+' if d >= 0 else ''
    better = (d > 0) == higher_better
    tag = '✓' if better else '✗'
    return f'{sign}{d:.4f} {tag}'

# RGB Reconstruction
b_ssim = np.mean(nyu_base_ssim); l_ssim = np.mean(nyu_lora_ssim)
b_psnr = np.mean(nyu_base_psnr); l_psnr = np.mean(nyu_lora_psnr)
print('{:<30} {:>10.4f} {:>10.4f} {:>8}'.format('RGB Recon NYU — SSIM ↑', b_ssim, l_ssim, delta_str(b_ssim, l_ssim)))
print('{:<30} {:>10.2f} {:>10.2f} {:>8}'.format('RGB Recon NYU — PSNR ↑', b_psnr, l_psnr, delta_str(b_psnr, l_psnr)))
b_ssim = np.mean(dio_base_ssim); l_ssim = np.mean(dio_lora_ssim)
print('{:<30} {:>10.4f} {:>10.4f} {:>8}'.format('RGB Recon DIODE — SSIM ↑', b_ssim, l_ssim, delta_str(b_ssim, l_ssim)))
print('-' * 65)

# Surface Normals
if 'norm_base' in dir() and len(norm_base) > 0:
    b_mae = norm_base.mean(); l_mae = norm_lora.mean()
    b_d1  = np.mean(norm_base<11.25); l_d1 = np.mean(norm_lora<11.25)
    print('{:<30} {:>10.2f} {:>10.2f} {:>8}'.format('Normals DIODE — MAE ↓', b_mae, l_mae, delta_str(l_mae, b_mae)))
    print('{:<30} {:>10.4f} {:>10.4f} {:>8}'.format('Normals DIODE — δ<11.25° ↑', b_d1, l_d1, delta_str(b_d1, l_d1)))
    print('-' * 65)

# Semantic Segmentation
b_miou = np.mean(seg_base); l_miou = np.mean(seg_lora)
print('{:<30} {:>10.4f} {:>10.4f} {:>8}'.format('Semseg NYU — mIoU ↑', b_miou, l_miou, delta_str(b_miou, l_miou)))
print('=' * 65)

# Save JSON
results = {
    'lora_weights': LORA_WEIGHTS,
    'rgb_recon_nyu':  {'ssim_base': float(np.mean(nyu_base_ssim)), 'ssim_lora': float(np.mean(nyu_lora_ssim)),
                       'psnr_base': float(np.mean(nyu_base_psnr)), 'psnr_lora': float(np.mean(nyu_lora_psnr))},
    'rgb_recon_diode':{'ssim_base': float(np.mean(dio_base_ssim)), 'ssim_lora': float(np.mean(dio_lora_ssim))},
    'semseg_nyu':     {'miou_base': float(b_miou), 'miou_lora': float(l_miou)},
}
if 'norm_base' in dir() and len(norm_base) > 0:
    results['normals_diode'] = {'mae_base': float(norm_base.mean()), 'mae_lora': float(norm_lora.mean()),
                                'd1_base': float(np.mean(norm_base<11.25)), 'd1_lora': float(np.mean(norm_lora<11.25))}
with open(os.path.join(OUTPUT_DIR, 'forgetting_results.json'), 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved → {OUTPUT_DIR}/forgetting_results.json')